In [ ]:
import sys
from functools import reduce


def power_of_four(y):
    return y ** 4


def is_not_positive(y):
    return y <= 0


def sum_filtered(nums):
    return reduce(lambda acc, y: acc + power_of_four(y), filter(is_not_positive, nums), 0)


def parse_case(x_line, y_line):
    x = int(x_line.strip())
    tokens = y_line.strip().split()
    if len(tokens) != x:
        return -1
    return sum_filtered(map(int, tokens))


def process_cases(pairs, results):
    if not pairs:
        return results
    x_line, y_line = pairs[0]
    return process_cases(pairs[1:], results + [parse_case(x_line, y_line)])


def pair_lines(lines, idx, pairs):
    if idx + 1 >= len(lines):
        return pairs
    return pair_lines(lines, idx + 2, pairs + [(lines[idx], lines[idx + 1])])


def main():
    raw = sys.stdin.read()
    non_empty = list(filter(lambda l: l.strip() != '', raw.splitlines()))
    n = int(non_empty[0].strip())
    pairs = pair_lines(non_empty[1:], 0, [])
    results = process_cases(pairs[:n], [])
    print('\n'.join(map(str, results)))


if __name__ == "__main__":
    main()

In [9]:
import hmac
import hashlib
import time
import struct
import base64
import requests
import json

# --- CONFIGURATION ---
USER_ID = "williamask112@gmail.com"
GITHUB_URL = "https://gist.github.com/Toxinityy/4ec399a6c0845a1f78bc1bfcaa51bb21"  # <-- fill this in
# ---------------------

def generate_hennge_totp(userid: str) -> str:
    secret_bytes = (userid + "HENNGECHALLENGE004").encode('utf-8')
    time_step = int(time.time() / 30)
    time_bytes = struct.pack('>q', time_step)
    hmac_hash = hmac.new(secret_bytes, time_bytes, hashlib.sha512).digest()
    offset = hmac_hash[-1] & 0x0F
    truncated_hash = struct.unpack_from('>I', hmac_hash, offset)[0]
    code = truncated_hash & 0x7FFFFFFF
    return str(code % 10**10).zfill(10)

def submit_challenge():
    totp = generate_hennge_totp(USER_ID)

    # Build Basic Auth header manually: base64(userid:password)
    credentials = f"{USER_ID}:{totp}"
    encoded = base64.b64encode(credentials.encode('utf-8')).decode('utf-8')

    headers = {
        "Content-Type": "application/json",
        "Authorization": f"Basic {encoded}"
    }

    payload = {
        "github_url": GITHUB_URL,
        "contact_email": USER_ID,
        "solution_language": "python"
    }

    response = requests.post(
        "https://api.challenge.hennge.com/challenges/backend-recursion/004",
        headers=headers,
        data=json.dumps(payload)  # serialize manually to avoid any surprises
    )

    print(f"Status Code: {response.status_code}")
    print(f"Response:    {response.text}")

if __name__ == "__main__":
    submit_challenge()

Status Code: 200
Response:    {"message":"Congratulations! You have achieved mission 3"}
